
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# Demo - Building and Logging a Retrieval Agent

## Overview

In this demo, we'll explore how to build and log a production-ready retrieval agent using Databricks Mosaic AI. A retrieval agent combines the power of large language models with your organization's knowledge base to provide accurate, context-aware responses. We'll walk through testing vector search in the AI Playground, building an agent with LangChain, implementing MLflow tracing for observability, and registering the agent as a model for deployment.

## Learning Objectives
By the end of this demo, you will be able to:
- **Test** vector search functionality using the AI Playground UI for rapid prototyping.
- **Build** a retrieval agent using LangChain with vector search index.
- **Implement** MLflow tracing to monitor and debug agent interactions.
- **Register** the agent as a model in Model Registry.

## Requirements:
- A pre-created **vector search endpoint**. This is pre-created for you.
- **Serverless Compute (environment version 5)**. Follow the instructions [here](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) to select the appropriate environment version.
- Basic familiarity with LangChain and retrieval-augmented generation (RAG) concepts.

## Setup

Run the code below to install required libraries and configure your classroom environment. This step ensures all dependencies are available and your workspace is ready for the demo.

In [0]:
%run ../Includes/Classroom-Setup-04 $section="demo"

In [0]:
pip install databricks-langchain

## A. Testing Vector Search in AI Playground

Before building our retrieval agent programmatically, we'll test the vector search index using the **AI Playground**. The AI Playground provides a **user-friendly interface for experimenting** with your vector search index, allowing you to quickly validate that your knowledge base is working correctly and understand how the retrieval system responds to different queries.

This interactive testing phase is valuable for understanding retrieval quality, identifying potential issues with your embeddings or chunking strategy, and refining your approach before investing time in code development.

**Dataset Information:** The vector search index contains data from a fictitious robot manufacturer for a robot named Orion. Documents include context-grounded answers sourced from internal design manuals, compliance documentation, and maintenance guides, enabling accurate retrieval of technical and regulatory information.

### A1. Configure Vector Search in Playground via Catalog Explorer

Databricks now streamlines the process of testing your vector search index in the AI Playground.

**Follow these steps to launch Playground with your vector search index:**

1. In your Databricks workspace, open **Catalog Explorer** and navigate to your vector search index.
   - Example: `{{{catalog_name}}}.{{{schema_name}}}.docs_chunked_index`

1. On the top right of the index details page, click the **Try in Playground** button.

1. The AI Playground will open automatically with your vector search index pre-configured as a retrieval tool.

1. Select your preferred large language model (LLM) in the Playground interface. We recommend using `GPT OSS 120B` model.

   * Click on **Use Endpoint**.

1. Start entering queries to test retrieval and response quality.

**Tip:** You could manually add your retrieval tool in the Playground too. But this method saves time as vector search is pre-configured, so you can select your LLM and start experimenting right away.

### A2. Test Queries and Experiment 

Now that your vector search is configured, we'll test it with various queries to evaluate retrieval quality and response accuracy.

**Follow these steps to test your retrieval system:**

1. In the chat interface, enter a question related to your knowledge base.
   - Example: *"How does the Orion motion controller maintain stability during high-speed movement?"*
   - Example: *"How does Orion verify compliance with ISO 13849-1?"*

1. Submit the query and observe the response from the language model.


**Experimenting Best Practices:**

1. **Examine the retrieved context:** Check which documents or chunks were retrieved and confirm their relevance to your question.

1. **Evaluate the response quality:** Ensure the answer accurately reflects your knowledge base and is grounded in the retrieved documents.

1. **Test edge cases:** Ask questions outside your knowledge base and try different phrasings to assess robustness.

1. **Iterate and refine:** Adjust retrieval parameters if results are poor and note patterns in what works well.


Once you're satisfied with the retrieval quality in the Playground, you're ready to build the agent programmatically in the next section.

## B. Building a Retrieval Agent with LangChain

In this section, we'll build a retrieval agent using **LangChain**, which is the framework in scope for this demo. LangChain offers robust tools for connecting language models to your organization's data, enabling context-aware responses and flexible agent workflows. **While we use LangChain here to demonstrate best practices, you can apply similar retrieval agent patterns with other frameworks or libraries that support large language models and vector search.** The concepts and architecture are transferable—choose the technology that best fits your production requirements.

We'll follow an **"agent as code"** approach by writing our agent implementation to a Python file (`agent.py`). This is the recommended method when logging the models with `mlflow`. The agent will use Unity Catalog's vector search as a tool, allowing it to dynamically retrieve relevant context when answering user questions.

### B1. Enable MLflow Tracing

Before we start building the agent, **let's enable MLflow tracing for LangChain** so we can observe agent inputs, tool usage, and outputs in detail.

MLflow offers robust tracing and observability for GenAI workflows, including LangChain, and supports many other frameworks and flavors. This broad integration allows you to monitor, debug, and analyze generative AI applications across diverse environments—all within a unified MLflow interface.

**💡 Note:** MLflow tracing (`autolog()`) is enabled by default in classic compute, but on serverless compute, you must enable it manually.



In [0]:
import mlflow
mlflow.langchain.autolog()

### B2. Create a LangChain Agent

In [0]:
llm_endpoint_name = "databricks-gpt-oss-120b"

In [0]:
from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langgraph.checkpoint.memory import InMemorySaver


def build_agent(llm_endpoint:str, index_name: str, num_results: int = 3):
    model = ChatDatabricks(
        endpoint=llm_endpoint,
        max_tokens=500,
    )

    vs_tool = VectorSearchRetrieverTool(
        name="orion_knowledge_search",
        index_name=index_name,
        description="Search Orion knowledge base for relevant information",
        num_results=num_results,
    )

    # Optional: use a in memory saver to save the agent's state
    checkpointer = InMemorySaver()

    system_prompt = """You are the Orion Knowledge Assistant (OKA). Respond in a clear, professional, and factual tone appropriate for engineers and technical staff. Use only verified information from Orion's internal documents, and include source references when available. If the answer cannot be found, clearly state that and suggest related sections or next steps. Do not speculate, make assumptions, or provide information outside the provided context."""

    agent = create_agent(
        model=model, 
        tools=[vs_tool], 
        system_prompt=system_prompt,
        checkpointer=checkpointer,
        )
    return agent

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "databricks-demo-4"}}

# Quick smoke test
agent = build_agent(llm_endpoint_name, vs_index_name, 3)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is Orion?"}]},
    config=config
)
print(response['messages'][-1].content)


### B3. Review MLflow Tracing UI

The MLflow Tracing UI provides a comprehensive view of your agent's execution and tool usage. The output above shows tracing UI. Alternatively, if you want to review traces of an experiment that you run before, you can view in the **Experiments** page.
To access traces of an experiment, select an experiment and navigate to the **Traces** tab within your experiment.

- The **Summary** tab displays high-level information for each trace, including inputs, outputs, and trace metadata.
- The **Details & Timeline** tab offers a breakdown of every step in the trace, showing all LLM invocations, tools called, results returned from tools, and the final generated output. This helps you understand the agent's reasoning and data flow.
- On the left side, clicking on the timeline icon, you can enable the execution of **timeline view** to visualize the duration of each step, making it easy to identify bottlenecks or performance issues.
- Selecting an individual trace reveals additional details on the right panel. If any errors occurred, you can inspect them in the **Events** tab, which lists error messages and relevant context.


Reviewing these tabs helps you validate agent behavior, debug issues, and optimize your agent development workflows with clear, actionable insights.

## C. Log the Agent to Model registry

In this section we will show to log a model to model registry. First, we need to abstract the agent code from this notebook by creating a file with all agent code. Also, as the agent code can be run in any environment, we will create a `.yaml` config file. This will be logged along with the agent code.

### C1. Create `agent-config` 

In [0]:
import yaml

def create_config(llm_endpoint_name: str, index_name: str, num_results: int = 3):
    """Create a minimal YAML config for the agent."""
    config = {
        "llm_endpoint_name": llm_endpoint_name,
        "vector_search": {
            "index_name": index_name,
            "num_results": num_results
        }
    }
    return config


# Create config file
llm_endpoint_name = "databricks-gpt-oss-120b"

agent_config = create_config(llm_endpoint_name, vs_index_name)

# Write YAML file (for agent.py to read later)
with open("agent-config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(agent_config, f, sort_keys=False)

print("✅ Config file written: agent-conf.yaml")
print(yaml.safe_dump(agent_config, sort_keys=False))


### C2. Write Agent Code to the File

We will create an `agent.py` file to be used when logging the agent. This file includes;
- Loading config file 
- LangChain code that we created in the previous step
- Predict and Response format based on mlflow API


In [0]:
%%writefile agent.py
import os
from uuid import uuid4
from typing import Any, Dict, List

import yaml
import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langgraph.checkpoint.memory import InMemorySaver

# Load agent configuration from YAML file
def _load_config(path: str = "agent-config.yaml") -> Dict[str, Any]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Config file not found at '{path}'")
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    llm_endpoint = cfg.get("llm_endpoint_name")
    vs = cfg.get("vector_search", {}) or {}
    index_name = vs.get("index_name")
    num_results = int(vs.get("num_results", 3))
    if not llm_endpoint or not index_name:
        raise ValueError("Missing 'llm_endpoint_name' or 'vector_search.index_name' in agent-config.yaml")
    return {
        "llm_endpoint_name": llm_endpoint,
        "vs_index_name": index_name,
        "vs_num_results": num_results,
    }

# Build LangChain agent with LLM and vector search tool
def build_agent(llm_endpoint: str, index_name: str, num_results: int = 3):
    model = ChatDatabricks(endpoint=llm_endpoint, max_tokens=500)
    vs_tool = VectorSearchRetrieverTool(
        name="orion_knowledge_search",
        index_name=index_name,
        description="Search Orion knowledge base for relevant information",
        num_results=num_results,
    )
    checkpointer = InMemorySaver()
    system_prompt = (
        "You are the Orion Knowledge Assistant (OKA). Respond in a clear, professional, and factual tone "
        "appropriate for engineers and technical staff. Use only verified information from Orion's internal "
        "documents, and include source references when available. If the answer cannot be found, clearly state "
        "that and suggest related sections or next steps. Do not speculate, make assumptions, or provide "
        "information outside the provided context."
    )
    agent = create_agent(
        model=model,
        tools=[vs_tool],
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )
    return agent

# Extract last user message from conversation
def _last_user_text(messages: List[Dict[str, Any]]) -> str:
    user_msgs = [m for m in messages if (m.get("role") == "user")]
    return str(user_msgs[-1].get("content", "")) if user_msgs else str(messages[-1].get("content", ""))

# MLflow ResponsesAgent implementation for LangChain agent
class LangChainResponsesAgent(ResponsesAgent):
    def __init__(self):
        cfg = _load_config()
        self._cfg = cfg
        self._agent = build_agent(
            llm_endpoint=cfg["llm_endpoint_name"],
            index_name=cfg["vs_index_name"],
            num_results=cfg["vs_num_results"],
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        msgs = [m.model_dump() for m in request.input]  # [{'role': 'user'|'assistant', 'content': '...'}, ...]
        _ = _last_user_text(msgs) if msgs else ""

        # Generate a unique thread ID for each prediction
        thread_id = f"oka-{uuid4()}"

        result = self._agent.invoke(
            {"messages": msgs},
            config={"configurable": {"thread_id": thread_id}},
        )
        # Extract agent response text
        try:
            text = result["messages"][-1].content
        except Exception:
            text = str(result)

        return ResponsesAgentResponse(
            output=[self.create_text_output_item(text, str(uuid4()))],
            custom_outputs=request.custom_inputs,
        )

# Set the model for mlflow. This is needed when using agent-as-code approach
AGENT = LangChainResponsesAgent()
mlflow.models.set_model(AGENT)

### C3. Import the Agent from File

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
from agent import AGENT as agent

mlflow.langchain.autolog()

response  = agent.predict(
    {"input": [{"role": "user", "content": "What is Orion?"}]}
)


## D. Logging and Registering the Model to Model registry

After loading and testing our agent we can go ahead and log the agent to model registry. This allows us to version the model, give aliases, tag and manage permissions. We can deploy the models from model registry using model serving. Note that the deployment of agents is not in the scope of this module.


### D1. Define Model Resources

Before logging the agent to MLflow, we need to define the **resources** that the agent depends on. Resources represent external dependencies such as vector search indexes, serving endpoints, tables, or functions that the agent uses during inference.

By explicitly declaring these resources, MLflow can:
* **Track dependencies** for reproducibility and lineage.
* **Validate availability** of required resources before deployment.
* **Enable proper permissions** and access control in production environments.

For our Orion Knowledge Assistant, we'll define two key resources:
1. **DatabricksVectorSearchIndex**: The vector search index containing Orion's knowledge base.
1. **DatabricksServingEndpoint**: The LLM endpoint used for generating responses.

These resources ensure that when the model is deployed, it has access to the necessary infrastructure components.

**🚨 Important:** Before proceeding, ensure you have imported the classroom setup config file. This step is required because the Python kernel was restarted in the previous section, and you need to reload the catalog and schema names for correct resource configuration.

In [0]:
%run ../Includes/Classroom-Setup-Common

In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint

# Redefine variables after Python restart
vs_index_name = f"{catalog}.{schema}.docs_chunked_index"
llm_endpoint_name = "databricks-gpt-oss-120b"

# Define the resources that the agent depends on
resources = [
    DatabricksVectorSearchIndex(index_name=vs_index_name),
    DatabricksServingEndpoint(endpoint_name=llm_endpoint_name)
]

print("Resources defined:")
for resource in resources:
    print(f"  - {resource}")

### D2. Log the Agent Model with MLflow

Now that we've defined our agent's resources, we'll **log the agent as a model** using MLflow. Logging captures the agent code, configuration, dependencies, and metadata in a structured format that can be versioned, tracked, and deployed.

When we log a model with MLflow, we create a **run** that records:
* **Model artifacts**: The agent code (`agent.py`) and configuration (`agent-config.yaml`).
* **Dependencies**: Python packages required to run the agent.
* **Resources**: External dependencies like vector search indexes and serving endpoints.
* **Input/output examples**: Sample data demonstrating the expected model interface.
* **Metadata and tags**: Information about the model version, purpose, and lineage.

This logged model becomes a **reproducible artifact** that can be loaded, tested, and deployed in any environment with the same dependencies and resources available.

In [0]:
import mlflow
from importlib.metadata import version as get_version

# Define model name and tags
model_name = "orion_knowledge_assistant"
tags_to_register = {
    "model_type": "retrieval_agent",
    "framework": "langchain",
    "use_case": "orion_knowledge_base"
}

# Create an input example for the model signature
input_example = {
    "input": [
        {"role": "user", "content": "What is Orion?"}
    ]
}

# Start an MLflow run and log the model
with mlflow.start_run():
    mlflow.set_tags(tags_to_register)
    
    logged_agent_info = mlflow.pyfunc.log_model(
        name=model_name,
        python_model="agent.py",
        code_paths=["agent-config.yaml"],
        input_example=input_example,
        pip_requirements=[
            f"databricks-vectorsearch=={get_version('databricks-vectorsearch')}",
            f"databricks-langchain=={get_version('databricks-langchain')}",
            f"langchain=={get_version('langchain')}",
            f"mlflow=={get_version('mlflow')}",
        ],
        resources=resources
    )
    
    # Save the model URI for later use
    model_uri = logged_agent_info.model_uri
    
print(f"✅ Model logged successfully!")
print(f"Model URI: {model_uri}")

### D3. Register the Model to Unity Catalog

Now that we've logged our agent model, we'll **register it to Unity Catalog's Model Registry**. While logging and registering may seem similar, they serve distinct purposes in the MLOps lifecycle.

**Understanding the Difference:**

* **Logging** creates a versioned artifact within an MLflow experiment run. It captures the model code, dependencies, and metadata at a specific point in time. Logged models are tied to individual runs and are primarily used for experimentation and development.

* **Registering** promotes a logged model to the Model Registry, making it a **managed, governed asset** with a unique name in Unity Catalog. Registered models support:
  - **Version management**: Track multiple versions of the same model.
  - **Aliases**: Assign labels like `Champion` or `Challenger` to specific versions.
  - **Governance**: Apply permissions, tags, and lineage tracking.
  - **Deployment**: Serve models directly from the registry to production endpoints.

By registering our agent to Unity Catalog, we transform it from an experimental artifact into a production-ready asset that can be discovered, governed, and deployed across the organization.

In [0]:
# Set the registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Define the fully qualified model name in Unity Catalog
UC_MODEL_NAME = f"{catalog}.{schema}.orion_knowledge_assistant"

# Register the model to Unity Catalog
uc_registered_model_info = mlflow.register_model(
    model_uri=model_uri, 
    name=UC_MODEL_NAME
)

print(f"✅ Model registered successfully to Unity Catalog!")
print(f"Model Name: {UC_MODEL_NAME}")
print(f"Version: {uc_registered_model_info.version}")

### D4. Test Model Inference

With our agent registered in Unity Catalog, we can now **load and test it** to verify that it works correctly. Loading a model from the registry ensures that we're using the exact version that was logged, with all its dependencies and configurations intact.

We'll demonstrate two approaches:
1. **Loading from the model URI**: Using the URI from the logging step.
1. **Loading from Unity Catalog**: Using the fully qualified model name.

Both methods allow us to invoke the agent with test inputs and validate that it retrieves relevant context from the vector search index and generates appropriate responses.

In [0]:
# Load the model from the model URI (pyfunc flavor)
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

# Use the input example that was logged with the model
input_data = pyfunc_model.input_example

print("Input data:")
print(input_data)
print("\n" + "="*50 + "\n")

# Make a prediction using the loaded model
result = mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_data,
    env_manager="uv",
)

print("Agent Response:")
print(result)

### D5. Explore the Model Registry UI

With our agent successfully registered in Unity Catalog, we can now **explore the Model Registry UI** to understand how to manage, monitor, and govern our model. The Model Registry provides a centralized interface for tracking model versions, lineage, artifacts, and performance.

**Follow these steps to explore your registered model:**

1. Navigate to your model in Catalog Explorer:
   - Go to **Catalog Explorer** in your Databricks workspace.
   - Navigate to `{{{catalog_name}}}.{{{schema_name}}}.orion_knowledge_assistant`.
   - Click on the most recent version of the model.

1. Explore the four main tabs:

   * **Overview**: This tab displays key information about your model version:
     - **Metrics**: Any metrics logged with the model during training or evaluation.
     - **Activity Log**: A chronological record of changes, updates, and deployments.
     - **Model Signature**: The input/output schema defining the expected data format.
     - **Version Information**: Details about this specific version, including creation date and creator.
     - **Active Endpoints**: Any serving endpoints currently using this model version.
     - **Tags**: Custom metadata tags for organization and discovery.

   * **Lineage**: Shows upstream sources and downstream consumers for model version tracking.

   * **Artifacts**: Lists all files and dependencies registered with the model.

   * **Traces**: Displays detailed execution traces for agent invocations and debugging.

The Model Registry UI provides a comprehensive view of your model's lifecycle, making it easy to manage versions, track lineage, and ensure governance across your organization.

## E. Summary

In this demo, we explored the complete workflow for building, logging, and registering a production-ready **retrieval agent** using Databricks Mosaic AI. We started by **testing vector search in the AI Playground**, then built a LangChain-based agent that combines LLMs with a knowledge base stored in Vector Search for context-aware responses. We utilized **MLflow tracing** for observability, adopted an **"agent as code"** approach with Python files and YAML configuration, and logged our agent to MLflow with explicit **resource dependencies**. Finally, we registered the agent to Unity Catalog's Model Registry, transforming it into a **governed, versioned asset ready for production deployment**.

**Key Takeaways:**

* **Test vector search indexes** in the **AI Playground** before writing code to validate retrieval quality and response accuracy.
* **Enable MLflow tracing** to monitor agent behavior, including tool usage, LLM calls, and response generation.
* **Adopt an "agent as code" approach** by abstracting logic into **Python files** with **YAML configuration** for portability and maintainability.
* **Define explicit resource dependencies** (vector search indexes, serving endpoints) when logging models to **MLflow**.
* **Register models to Unity Catalog's Model Registry** to enable version management, governance, lineage tracking, and production deployment.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>